In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from picoscenes import Picoscenes
from pathlib import Path
from picoparser import PicoParser

In [3]:
filename = "1/rx_2_260413_161147"
csi_path = Path(f"/home/drone/BTP/harsh_data/{filename}.csi")

In [4]:
def extract_csi_parallely(
    csi_path: Path,
    ts_start: int,
    ts_end: int,
    tx_idx: int   = 0,
    rx_idx: int   = 0,
    csi_idx: int  = 0,
    min_len: int  = 998,
    num_workers: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """Parse a .csi file and return (ts_arr [ns, int64], csi_arr [complex64], bw_arr [int])."""
    with PicoParser(csi_path, num_workers) as parser:
        i = 0
        ts_list, csi_list, bw_list = [], [], []

        for fr in parser.getFrames():
            try:
                ts  = fr.rxSBasic.systemTime
                csi = fr.csi.csi
                csi = np.array([sub_csi[tx_idx][rx_idx][csi_idx] for sub_csi in csi])
                bw = fr.csi.cbw
            except Exception:
                continue

            if ts is None:
                continue
            if (int(ts) < int(ts_start)):
                continue
            if (int(ts) > int(ts_end)):
                break
            if csi.ndim == 0:
                continue
            if csi.shape[0] < min_len:
                continue

            ts_list.append(int(ts))
            csi_list.append(csi[:min_len])
            bw_list.append(bw)

            if (i + 1) % 1000 == 0:
                print(f"    ...parsed {i + 1} frames")
            i += 1

        print(f"    {i} frames parsed in total")
        return np.array(ts_list, dtype=np.int64), np.array(csi_list), np.array(bw_list)

In [5]:
ts_arr, csi_arr, bw_arr = extract_csi_parallely(csi_path, 1776076912050000000, 1776076972391000000, num_workers=4)

    ...parsed 1000 frames
    ...parsed 2000 frames
    ...parsed 3000 frames
    ...parsed 4000 frames
    ...parsed 5000 frames
    ...parsed 6000 frames
    ...parsed 7000 frames
    ...parsed 8000 frames
    ...parsed 9000 frames
    ...parsed 10000 frames
    ...parsed 11000 frames
    ...parsed 12000 frames
    ...parsed 13000 frames
    ...parsed 14000 frames
    ...parsed 15000 frames
    ...parsed 16000 frames
    ...parsed 17000 frames
    ...parsed 18000 frames
    ...parsed 19000 frames
    ...parsed 20000 frames
    ...parsed 21000 frames
    ...parsed 22000 frames
    ...parsed 23000 frames
    ...parsed 24000 frames
    ...parsed 25000 frames
    ...parsed 26000 frames
    ...parsed 27000 frames
    ...parsed 28000 frames
    28825 frames parsed in total


In [6]:
print(f"CSI array dtype: {csi_arr.dtype}, Timestamps array dtype: {ts_arr.dtype}")

print(f"CSI array shape: {csi_arr.shape}")
print(f"Timestamps array shape: {ts_arr.shape}")

# Data Frequency
time_diffs = np.diff(ts_arr) / 1e9  # Convert ns to seconds
total_time = (ts_arr[-1] - ts_arr[0]) / 1e9
csi_freq = len(ts_arr)/total_time
print(f"Total time span: {total_time:.3f} seconds")
print(f"Average time between frames: {np.mean(time_diffs):.3f} seconds")
print(f"Frequency: {csi_freq:.3f} Hz")

# get bandwidth of signal
from collections import Counter
import statistics
print(f"Bandwidth of different frames: {Counter(bw_arr)}")
print(f"Mode bandwidth = {statistics.mode(bw_arr)}")

CSI array dtype: complex64, Timestamps array dtype: int64
CSI array shape: (28825, 998)
Timestamps array shape: (28825,)
Total time span: 60.340 seconds
Average time between frames: 0.002 seconds
Frequency: 477.713 Hz
Bandwidth of different frames: Counter({np.int64(80): 28825})
Mode bandwidth = 80


In [ ]:
from scipy.signal.windows import hann

def generate_doppler_stacks(X, fs, win_len=256, hop=30, nfft=256):
    """
    X: complex CSI (T, S)
    returns: (num_frames, freq_bins, S)
    """
    T, S = X.shape

    # =========================
    # PHASE PROCESSING (correct)
    # =========================
    phase = np.abs(X)
    # phase =phase.T
    # phase = np.unwrap(phase, axis=0)

    # # remove DC / drift per subcarrier
    # phase = phase - np.mean(phase, axis=0, keepdims=True)

    num_frames = (T - win_len) // hop + 1
    window = np.hanning(win_len)

    stacks = []

    for i in range(num_frames):
        start = i * hop
        segment = phase[start:start + win_len]   # (win_len, S)

        if segment.shape[0] < win_len:
            break

        segment = segment * window[:, None]
        hann_window= np.expand_dims(hann(256), axis=-1)
        segment = np.multiply(segment, hann_window)
        # Is multiplying with hann window twice intentional or a bug?

        # FFT along time axis (THIS matches your code)
        S_fft = np.fft.fft(segment, n=nfft, axis=0)     # applying fft on real vector makes sense? SHARP applies it on the processed complex CSI matrix
        S_fft = np.fft.fftshift(S_fft, axes=0)

        stacks.append(np.abs(S_fft * np.conjugate(S_fft)))
          # (freq, S)

    stacks = np.array(stacks)  # (frames, freq, S)

    # =========================
    # Frequency axis (same as yours)
    # =========================
    freqs = np.fft.fftshift(np.fft.fftfreq(nfft, d=1/fs))

    return stacks, freqs

In [7]:
from scipy.signal.windows import hann

def generate_doppler_stacks(X, fs, win_len=256, hop=30, nfft=256):
    """
    X: complex CSI (T, S)
    returns: (num_frames, freq_bins, S)
    """
    T, S = X.shape

    amp = np.abs(X)
    
    num_frames = (T - win_len) // hop + 1

    stacks = []

    for i in range(num_frames):
        start = i * hop
        segment = amp[start:start + win_len]   # (win_len, S)

        if segment.shape[0] < win_len:
            break

        hann_window= np.expand_dims(hann(256), axis=-1)
        segment = np.multiply(segment, hann_window)

        S_fft = np.fft.fft(segment, n=nfft, axis=0)     # applying fft on real vector makes sense? SHARP applies it on the processed complex CSI matrix
        S_fft = np.fft.fftshift(S_fft, axes=0)

        stacks.append(np.abs(S_fft * np.conjugate(S_fft)))

    stacks = np.array(stacks)  # (frames, freq, S)

    freqs = np.fft.fftshift(np.fft.fftfreq(nfft, d=1/fs))

    return stacks, freqs

In [ ]:
import cv2

def create_doppler_video(stacks, freqs, output_file="doppler.mp4", fps=10):
    """
    stacks: (frames, freq, S)
    """

    frames, F, S = stacks.shape

    # Normalize for visualization
    stacks_db = 10 * np.log10(stacks + 1e-9)

    # scale to 0–255
    stacks_norm = (stacks_db - stacks_db.min()) / (stacks_db.max() - stacks_db.min())
    stacks_norm = (stacks_norm * 255).astype(np.uint8)

    height, width = F, S

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_file, fourcc, fps, (width, height), isColor=True)

    for i in range(frames):
        frame = stacks_norm[i]

        # apply colormap (jet-like)
        frame_color = cv2.applyColorMap(frame, cv2.COLORMAP_JET)

        out.write(frame_color)

    out.release()
    print(f"Video saved to {output_file}")

In [11]:
stacks, freqs = generate_doppler_stacks(csi_arr, csi_freq)

In [12]:
create_doppler_video(stacks, freqs, output_file="doppler4_drone.mp4", fps=10)

Video saved to doppler4_drone.mp4


In [ ]:
def plot_collapsed_doppler_spectrogram(csi_profile_d_arr, db=True):
    """
    stacks: (frames, freq, S)
    freqs : (freq,)

    Steps:
    1. Sum across subcarriers (axis=2)
    2. Stack frames to form (freq × time)
    3. Plot spectrogram
    """

    spec = csi_profile_d_arr.T

    # =========================
    # Step 6: plot
    # =========================
    print(spec.shape)
    plt.figure(figsize=(10, 6))

    plt.imshow(
        spec,
        aspect='auto',
        cmap='jet',
        # extent=[time_axis[0], time_axis[-1], freqs_plot[0], freqs_plot[-1]]
    )

    plt.xlabel("Time (frames)")
    plt.ylabel("Doppler Frequency (Hz)")
    plt.title("Collapsed Doppler Spectrogram")

    plt.colorbar(label="Magnitude (dB)" if db else "Magnitude")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_collapsed_doppler_spectrogram(csi_profile_d_arr)

In [ ]:
def get_amplitude_matrix(df : pd.DataFrame) -> pd.DataFrame:
    df_csi = df.drop(columns=['timestamp'])
    amp_values = np.abs(df_csi.to_numpy())
    ts = df['timestamp'].to_numpy()
    ts = (ts - ts[0]) * 1e-9   # ns → seconds, relative to start
    ts = ts.reshape(-1, 1)
    amp_matrix = pd.DataFrame(np.hstack((ts, amp_values)))
    amp_matrix.columns = ['timestamp'] + list(range(1, amp_values.shape[1] + 1))
    return amp_matrix

from scipy.signal import get_window
from scipy.fft import fft
from scipy.interpolate import interp1d

TARGET_LEN  = 100             # interpolate each window to this many time steps
SAMPLE_RATE = 100             # assumed CSI packet rate (Hz)
WINDOW_NS   = 200_000_000       # 2 ms in nanoseconds
STRIDE_NS   = 100_000_000       # 1 ms in nanoseconds
MIN_FRAMES  = 20

def _compute_doppler(csi: np.ndarray) -> np.ndarray:
    """complex (T, S) → float32 (T//2, S)"""
    amp = np.abs(csi)
    T   = amp.shape[0]
    win = get_window("hann", T)
    spec = np.abs(fft(amp * win[:, None], axis=0))
    return spec[:T // 2].astype(np.float32)


def _normalize(x: np.ndarray) -> np.ndarray:
    mu, sigma = x.mean(), x.std()
    return (x - mu) / (sigma + 1e-8)


def _interpolate(csi: np.ndarray, ts_ns: np.ndarray) -> np.ndarray:
    """complex (N, S) + timestamps → complex (TARGET_LEN, S)"""
    t_range = ts_ns[-1] - ts_ns[0]
    t_old = (ts_ns - ts_ns[0]) / t_range if t_range > 0 else np.linspace(0, 1, len(ts_ns))
    t_new = np.linspace(0, 1, TARGET_LEN)
    rf = interp1d(t_old, np.real(csi), axis=0, kind="linear", fill_value="extrapolate")
    if_ = interp1d(t_old, np.imag(csi), axis=0, kind="linear", fill_value="extrapolate")
    return (rf(t_new) + 1j * if_(t_new)).astype(np.complex64)


def raw_csi_to_doppler_maps(csi: np.ndarray, ts_ns: np.ndarray) -> np.ndarray:
    """
    Slide windows over raw CSI and return stacked Doppler maps.
    csi   : complex (N_frames, 1001)
    ts_ns : int64   (N_frames,) nanoseconds
    returns float32 (W, 1, T//2, 1001)
    """
    maps = []
    t0 = ts_ns[0]
    while t0 + WINDOW_NS <= ts_ns[-1] + STRIDE_NS:
        mask = (ts_ns >= t0) & (ts_ns < t0 + WINDOW_NS)
        if mask.sum() >= MIN_FRAMES:
            try:
                csi_interp = _interpolate(csi[mask], ts_ns[mask])
                d = _normalize(_compute_doppler(csi_interp))
                maps.append(d[np.newaxis])  # (1, T//2, 1001)
            except Exception as e:
                print(f"  [WARN] window at t={t0/1e9:.1f}s failed: {e}")
        t0 += STRIDE_NS
    if not maps:
        return np.empty((0, 1, TARGET_LEN // 2, csi.shape[1]), dtype=np.float32)
    return np.stack(maps, axis=0)  # (W, 1, T//2, S)

In [ ]:
import numpy as np
import cv2

def doppler_maps_to_video(
    maps: np.ndarray,
    out_path: str = "doppler.mp4",
    fps: float = 1.0,
    resize: tuple = (800, 600)  # (width, height)
):
    """
    maps: (W, 1, F, S)
    Creates a video where each frame is an F x S spectrogram.
    """

    if maps.shape[0] == 0:
        raise ValueError("Empty input maps")

    W, _, F, S = maps.shape

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video = cv2.VideoWriter(out_path, fourcc, fps, resize)

    for i in range(W):
        frame = maps[i, 0]  # (F, S)

        # Normalize to 0–255 for visualization
        frame_norm = frame - frame.min()
        if frame_norm.max() > 0:
            frame_norm = frame_norm / frame_norm.max()
        frame_uint8 = (frame_norm * 255).astype(np.uint8)

        # Apply colormap for better visibility
        frame_color = cv2.applyColorMap(frame_uint8, cv2.COLORMAP_JET)

        # Resize for video
        frame_resized = cv2.resize(frame_color, resize)

        video.write(frame_resized)

    video.release()
    print(f"Video saved to {out_path}")

In [ ]:
doppler_maps = raw_csi_to_doppler_maps(csi_arr, ts_arr)

In [ ]:
doppler_maps_to_video(doppler_maps, fps=10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_spectrogram(doppler_maps: np.ndarray, freq_bin: int):
    """
    doppler_maps: (W, 1, F, S)
    """

    # 1. Remove channel dim → (W, F, S)
    x = doppler_maps[:, 0, :, :]

    # 2. Remove DC component (VERY important)
    x = x[:, 0:, :]   # (W, F, S)

    # 3. Average over subcarriers → (W, F-1)
    # spec = x.mean(axis=2)
    spec = x[:, freq_bin, : ]

    # 4. Transpose → (F-1, W)
    spec = spec.T

    # 5. Log scale (mandatory)
    spec_log = np.log1p(spec)

    # 6. Optional: clip extreme values
    spec_log = np.clip(spec_log, 0, np.percentile(spec_log, 99))

    # 7. Plot
    plt.figure(figsize=(10, 6))

    plt.imshow(
        spec_log,
        aspect='auto',
        origin='lower',
        interpolation='nearest'
    )

    plt.colorbar(label="Log Magnitude")
    plt.xlabel("Time (windows)")
    plt.ylabel("Subcarrier Index")
    plt.title("Doppler Spectrogram")

    plt.tight_layout()
    plt.show()

In [ ]:
doppler_maps = raw_csi_to_doppler_maps(csi_arr, ts_arr)
df_maps = pd.DataFrame(doppler_maps[:, 0, :, :][:, 2:, :].mean(axis=2).T)
df_maps.to_csv('doppler_maps.csv')

In [ ]:
plot_spectrogram(doppler_maps, 49)

In [ ]:
def plot_doppler_spectrogram(doppler: np.ndarray):
    # Log scale for better visualization (very important!)
    doppler_log = np.log1p(doppler)
    doppler_log = doppler

    plt.figure(figsize=(10, 6))

    plt.imshow(
        doppler_log,
        aspect='auto',
        origin='lower',   # low freq at bottom
    )

    plt.colorbar(label="Log Magnitude")
    plt.xlabel("Subcarrier Index")
    plt.ylabel("Doppler Frequency Bin")
    plt.title("Doppler Spectrogram")

    plt.tight_layout()
    plt.show()

plot_doppler_spectrogram(doppler_amp_matrix)

In [ ]:
def plot_csi_spectrogram(amp_df: pd.DataFrame, title="CSI Amplitude Spectrogram"):
    """Preview helper: plot the full amplitude spectrogram for the entire recording."""
    fig, ax = plt.subplots(figsize=(24, 6))
    ts = amp_df['timestamp'].to_numpy()
    data = amp_df.drop(columns=['timestamp']).to_numpy()  # shape (frames, subcarriers)
    im = ax.imshow(data.T, aspect='auto', origin='lower', cmap='jet',
                   extent=[ts[0], ts[-1], 1, data.shape[1]])
    fig.colorbar(im, ax=ax, label='Amplitude')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Subcarrier Index')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def save_sliding_window_spectrograms(
    amp_df: pd.DataFrame,
    output_dir: str,
    window_sec: float = 2.0,
    stride_sec: float = 1.0,
    img_size: tuple[int, int] = (224, 224),
    cmap: str = 'jet',
    video_name: str = 'video',
) -> list[dict]:
    """
    Slide a window of `window_sec` seconds over `amp_df` in steps of
    `stride_sec` seconds, saving one model-ready spectrogram PNG per step.

    Each PNG covers the interval  [t_end - window_sec,  t_end)  where
    t_end = stride_sec * k  for k = 1, 2, 3, ...
    Starting at k=1 means the very first image covers [0, 2) seconds,
    which requires at least `window_sec` of data to exist.

    Images are saved without any axes, labels, titles or whitespace so
    they can be fed directly into the CNN as pixel arrays.

    Parameters
    ----------
    amp_df      : amplitude DataFrame from get_amplitude_matrix().
                  Must have a float 'timestamp' column in seconds.
    output_dir  : folder to write PNGs into (created if absent).
    window_sec  : width of the sliding window in seconds (default 2).
    stride_sec  : step between consecutive windows in seconds (default 1).
    img_size    : (width, height) in pixels of saved PNGs (default 224x224).
    cmap        : matplotlib colormap (default 'jet').
    video_name  : prefix used in output filenames.

    Returns
    -------
    List of dicts with keys: 'filename', 't_start', 't_end', 'second'
    (one entry per saved PNG — useful for merging with label CSVs).
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    ts  = amp_df['timestamp'].to_numpy()          # (N,)  seconds, float
    amp = amp_df.drop(columns=['timestamp']).to_numpy()  # (N, subcarriers)

    total_duration = ts[-1]                        # seconds
    dpi = 72                                       # screen-quality is enough
    fig_w = img_size[0] / dpi
    fig_h = img_size[1] / dpi

    # Pre-compute global amplitude range for consistent colour scale
    vmin, vmax = amp.min(), amp.max()

    metadata = []
    second = 1                                     # 1-based second index
    t_end = window_sec                             # first window ends here

    while t_end <= total_duration + 1e-9:          # small epsilon for float safety
        t_start = t_end - window_sec

        # Boolean mask for frames inside [t_start, t_end)
        mask = (ts >= t_start) & (ts < t_end)
        window_amp = amp[mask]                     # shape (frames_in_window, subcarriers)

        if window_amp.shape[0] == 0:
            print(f"  [skip] second={second:04d}  no frames in [{t_start:.2f}, {t_end:.2f})")
            second += 1
            t_end   += stride_sec
            continue

        # ── Draw the spectrogram ──────────────────────────────────────────
        fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)
        ax  = fig.add_axes([0, 0, 1, 1])           # fill entire figure, no margins
        ax.imshow(
            window_amp.T,                           # (subcarriers, frames) → correct orientation
            aspect='auto',
            origin='lower',
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation='nearest',
        )
        ax.axis('off')                              # no ticks, labels, or border

        fname    = f"{second}.png"
        out_path = os.path.join(output_dir, fname)
        fig.savefig(out_path, dpi=dpi, bbox_inches=None, pad_inches=0)
        plt.close(fig)

        metadata.append({
            'filename': fname,
            't_start' : round(t_start, 6),
            't_end'   : round(t_end,   6),
            'second'  : second,
        })

        if second % 10 == 0:
            print(f"  saved second={second:04d}  [{t_start:.2f}s, {t_end:.2f}s)  "
                  f"frames={window_amp.shape[0]}")

        second += 1
        t_end  += stride_sec

    print(f"\nDone. {len(metadata)} spectrograms saved to '{output_dir}'")
    return metadata

In [ ]:
# ── Preview: full spectrogram of the entire recording ─────────────────────────
plot_csi_spectrogram(doppler_amp_matrix)

# ── Generate sliding-window spectrograms (model inputs) ───────────────────────
# Each PNG = 2-second window, one per second, saved as 224×224 pixels.
# 'second' in the metadata aligns with the 'second' column in the label CSVs.
# meta = save_sliding_window_spectrograms(
#     amp_df      = amp_matrix,
#     output_dir  = f"model_data/{filename.split('/')[0]}",
#     window_sec  = 2.0,
#     stride_sec  = 1.0,
#     img_size    = (224, 224),
#     cmap        = 'jet',
#     video_name  = Path(filename).name,
# )

# # Inspect the metadata table
# meta_df = pd.DataFrame(meta)
# display(meta_df)